# 05. Risk Index Baseline
**목적**: Weather Hazard / Spatial Exposure / Facility Exposure / Historical Prior를 결합하여
설비별 화재위험도 (0~100점) 와 4단계 등급을 산출한다.

산출물:
- `final_risk` — 0~100 최종 위험도
- `risk_grade` — Low / Moderate / High / Very High
- `risk_grade_quantile` — 분위수 기준 등급 (비교용)
- 가중치 5개 시나리오별 결과 저장

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

from config import (DATA_PROCESSED, OUT_TABLES, WEIGHTS, GRADE_THRESHOLDS, TOP_K_PCTS)

In [ ]:
with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FC = cmap['facility']
FID_COL = FC['facility_id']

# 각 레이어 feature 로드
df_weather = pd.read_parquet(DATA_PROCESSED / 'weather_features.parquet')
df_buf     = pd.read_parquet(DATA_PROCESSED / 'buffer_features.parquet')
gdf_fac    = gpd.read_file(DATA_PROCESSED / 'facility_proj.gpkg')

print('Weather features:', df_weather.shape)
print('Buffer features:', df_buf.shape)
print('Facilities:', len(gdf_fac))

## 1. Spatial / Forest Exposure Score

In [ ]:
def percentile_score(series, ascending=True):
    s = series.fillna(series.median())
    rank = s.rank(pct=True, method='average')
    return (rank * 100).clip(0, 100) if ascending else ((1 - rank) * 100).clip(0, 100)


df_spatial = df_buf[[FID_COL]].copy()

# ── 산림 score ─────────────────────────────────────────────────────────────
has_forest = ('forest_ratio_500m' in df_buf.columns and
              df_buf['forest_ratio_500m'].notna().sum() > 0)

if has_forest:
    forest_score = (
        0.35 * percentile_score(df_buf['forest_ratio_500m']) +
        0.25 * percentile_score(df_buf.get('forest_ratio_1000m', pd.Series(0, index=df_buf.index))) +
        0.20 * percentile_score(df_buf.get('conifer_ratio_500m', pd.Series(0, index=df_buf.index))) +
        0.20 * percentile_score(df_buf.get('distance_to_forest_m', pd.Series(1000, index=df_buf.index)), ascending=False)
    )
    print('✓ 임상도 feature 적용')
else:
    forest_score = pd.Series(50.0, index=df_buf.index)
    print('⚠ 임상도 미수집 — 중립값 50 적용')

# ── 지형 score (slope 최우선) ──────────────────────────────────────────────
has_slope = ('slope_mean_500m' in df_buf.columns and
             df_buf['slope_mean_500m'].notna().sum() > 0)

if has_slope:
    # slope: 경사 클수록 화재 확산 빠름 (국제 연구 SHAP 1위)
    slope_score   = percentile_score(df_buf['slope_mean_500m'], ascending=True)
    # aspect: 남향(1.0)일수록 건조 → 위험
    aspect_score  = percentile_score(df_buf.get('aspect_risk_500m', pd.Series(0.5, index=df_buf.index)), ascending=True)
    terrain_score = 0.70 * slope_score + 0.30 * aspect_score
    print('✓ DEM(slope/aspect) feature 적용')
else:
    terrain_score = pd.Series(50.0, index=df_buf.index)
    print('⚠ DEM 미처리 — 중립값 50 적용 (dem.tif 존재, 전처리 필요)')

# ── 소방청 전기화재 prior score ────────────────────────────────────────────
has_elec = ('elec_fire_count_1000m' in df_buf.columns and
            df_buf['elec_fire_count_1000m'].notna().sum() > 0)

if has_elec:
    elec_score = percentile_score(
        df_buf['elec_fire_count_1000m'].fillna(0), ascending=True
    ) * 0.5 + percentile_score(
        df_buf.get('elec_fire_count_500m', pd.Series(0, index=df_buf.index)).fillna(0)
    ) * 0.5
    print('✓ 소방청 전기화재 prior 적용')
else:
    elec_score = pd.Series(0.0, index=df_buf.index)
    print('⚠ 소방청 데이터 미수집 — data/external/DOWNLOAD_GUIDE.md 참조')

# ── 최종 Spatial Exposure Score ───────────────────────────────────────────
# 가중치: 산림 40% + 지형(slope) 35% + 전기화재 prior 25%
df_spatial['spatial_exposure'] = (
    0.40 * forest_score +
    0.35 * terrain_score +
    0.25 * elec_score
).clip(0, 100)

print('\nSpatial Exposure Score 생성 완료')
print(df_spatial['spatial_exposure'].describe())

## 2. Facility Exposure Score

In [ ]:
df_fac_score = gdf_fac[[FID_COL]].copy()

# 설비 밀도 (buffer feature에서)
density_score = percentile_score(
    df_buf.get('facility_density_1000m', pd.Series(0, index=df_buf.index))
)

# 최근접 설비 거리: 가까울수록 노출도 높음 (ascending=False)
dist_score = percentile_score(
    df_buf.get('nearest_facility_dist_m', pd.Series(1000, index=df_buf.index)),
    ascending=False
)

# 설비 노후도 (연식 있는 경우)
if FC['install_year'] and FC['install_year'] in gdf_fac.columns:
    gdf_fac['facility_age'] = 2026 - gdf_fac[FC['install_year']]
    age_series = gdf_fac['facility_age'].fillna(gdf_fac['facility_age'].median())
    age_score = percentile_score(age_series, ascending=True)
else:
    age_score = pd.Series(50.0, index=gdf_fac.index)

df_fac_score = df_fac_score.copy()
df_fac_score.index = df_buf.index
df_fac_score['facility_exposure'] = (
    0.40 * density_score +
    0.35 * dist_score +
    0.25 * age_score.values
).clip(0, 100)

print('Facility Exposure Score 생성 완료')
print(df_fac_score['facility_exposure'].describe())

## 3. Historical Fire Prior Score

In [ ]:
df_prior_score = df_buf[[FID_COL]].copy()

# 산불통계 prior
has_forest_fire = ('fire_count_3000m' in df_buf.columns and
                   df_buf['fire_count_3000m'].notna().sum() > 0)

if has_forest_fire:
    forest_prior = (
        0.50 * percentile_score(df_buf['fire_count_5000m'].fillna(0)) +
        0.30 * percentile_score(df_buf['fire_count_3000m'].fillna(0)) +
        0.20 * percentile_score(df_buf['fire_count_1000m'].fillna(0))
    )
    print('✓ 산불발생통계 prior 적용')
else:
    forest_prior = pd.Series(0.0, index=df_buf.index)
    print('⚠ 산불통계 미수집 — data/external/DOWNLOAD_GUIDE.md 참조')

# 소방청 전기화재 prior (더 직접적인 evidence)
has_elec_fire = ('elec_fire_count_1000m' in df_buf.columns and
                 df_buf['elec_fire_count_1000m'].notna().sum() > 0)

if has_elec_fire:
    elec_prior = percentile_score(df_buf['elec_fire_count_1000m'].fillna(0))
    print('✓ 소방청 전기화재 prior 적용')
else:
    elec_prior = pd.Series(0.0, index=df_buf.index)

# 결합 (산불:전기화재 = 6:4)
df_prior_score['hist_prior'] = (
    0.60 * forest_prior + 0.40 * elec_prior
).clip(0, 100)

print('Historical Prior Score 생성 완료')
print(df_prior_score['hist_prior'].describe())

## 4. 전체 데이터 통합 (facility × date)

In [ ]:
# facility-level score (날짜 무관)
df_fac_all = df_buf[[FID_COL]].copy()
df_fac_all = df_fac_all.merge(df_spatial[[FID_COL, 'spatial_exposure']], on=FID_COL, how='left')
df_fac_all = df_fac_all.merge(df_fac_score[[FID_COL, 'facility_exposure']], on=FID_COL, how='left')
df_fac_all = df_fac_all.merge(df_prior_score[[FID_COL, 'hist_prior']], on=FID_COL, how='left')

# 날짜별 weather hazard 와 cross join
df_main = df_weather[[FID_COL, 'date', 'weather_hazard']].merge(
    df_fac_all, on=FID_COL, how='left'
)

print(f'통합 데이터: {df_main.shape}')
df_main.head()

## 5. Final Risk Score — 5개 가중치 시나리오

In [ ]:
def compute_risk(df, w):
    return (
        w['weather']  * df['weather_hazard'].fillna(50) +
        w['spatial']  * df['spatial_exposure'].fillna(50) +
        w['facility'] * df['facility_exposure'].fillna(50) +
        w['prior']    * df['hist_prior'].fillna(0)
    ).clip(0, 100)


scenario_results = {}
for scenario, w in WEIGHTS.items():
    df_main[f'final_risk_{scenario}'] = compute_risk(df_main, w)
    scenario_results[scenario] = df_main[f'final_risk_{scenario}'].describe()
    print(f'  {scenario}: mean={df_main[f"final_risk_{scenario}"].mean():.1f}, '
          f'std={df_main[f"final_risk_{scenario}"].std():.1f}')

# 기본 시나리오를 final_risk 로 고정
df_main['final_risk'] = df_main['final_risk_base']

## 6. 위험등급화

In [ ]:
def assign_grade_fixed(score, thresholds):
    for grade, (lo, hi) in thresholds.items():
        if lo <= score <= hi:
            return grade
    return 'Low'


def assign_grade_quantile(score_series):
    q80 = score_series.quantile(0.80)
    q60 = score_series.quantile(0.60)
    q30 = score_series.quantile(0.30)
    return pd.cut(
        score_series,
        bins=[-np.inf, q30, q60, q80, np.inf],
        labels=['Low', 'Moderate', 'High', 'Very High']
    )


# 고정 임계값
df_main['risk_grade'] = df_main['final_risk'].apply(
    lambda x: assign_grade_fixed(x, GRADE_THRESHOLDS)
)

# 분위수 기준 (비교용)
df_main['risk_grade_quantile'] = assign_grade_quantile(df_main['final_risk'])

print('등급 분포 (고정 임계값):')
print(df_main['risk_grade'].value_counts())
print('\n등급 분포 (분위수):')
print(df_main['risk_grade_quantile'].value_counts())

## 7. 저장

In [ ]:
OUT_TABLES.mkdir(parents=True, exist_ok=True)

df_main.to_parquet(DATA_PROCESSED / 'risk_scores.parquet', index=False)

# 가중치 시나리오 비교 테이블
pd.DataFrame(scenario_results).T.to_csv(OUT_TABLES / 'sensitivity_scenarios.csv')

# 최신 날짜 기준 Top-K 설비 테이블
latest_date = df_main['date'].max()
df_latest = df_main[df_main['date'] == latest_date].sort_values('final_risk', ascending=False)

for pct in TOP_K_PCTS:
    k = max(1, int(len(df_latest) * pct))
    top_k = df_latest.head(k)[[
        FID_COL, 'date', 'final_risk', 'risk_grade',
        'weather_hazard', 'spatial_exposure', 'facility_exposure', 'hist_prior'
    ]]
    top_k.to_csv(OUT_TABLES / f'top_{int(pct*100)}pct_facilities.csv', index=False)
    print(f'Top {int(pct*100)}% ({k}개) 저장')

print('\n저장 완료')
print('다음 단계: 06_modeling.ipynb')